In [ ]:
# Concatenation without batch correction, clustering and Palantir pseudotime.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
import scvelo as scv
import palantir as pl
import cellrank as cr

In [ ]:
!pwd

In [ ]:
sc.settings.figdir = "/home/yanjun2/scanpy/7"

In [ ]:
import glob
h5ad_files = glob.glob("/home/yanjun2/scanpy/4/*.h5ad")
adatas = [sc.read_h5ad(f) for f in h5ad_files]

In [ ]:
for adata in adatas:
    del adata.obs['leiden_res_1_5'] 
    del adata.obs['leiden_res_1'] 
    del adata.obs['leiden_res_0_6'] 
    adata.obsm = {}

In [ ]:
adata = ad.concat(adatas, join='outer', index_unique=None)

In [ ]:
adata

In [ ]:
 del adatas

In [ ]:
# Normalise, HVG and PCA
adata.X = adata.layers["counts"].toarray().copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["data"] = adata.X.copy()

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3)
sc.tl.pca(adata)
sc.pp.scale(adata)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=['pre_annotation'], legend_loc = 'right margin', ncols = 2, size = 100, save = 'pre_anno.png'
          )

In [ ]:
sc.pl.umap(adata, color=['timepoint'], legend_loc = 'right margin', ncols = 2,  save = 'timepoint.png'
          )

In [ ]:
sc.tl.leiden(
    adata,
    resolution= 1,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_1'
)

In [ ]:
sc.pl.umap(adata, color=['leiden_res_1'], legend_loc = 'right margin', ncols = 2,  save = 'leiden_res_1.png'
          )

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/4/integrated.h5ad')

In [ ]:
sc.tl.leiden(
    adata,
    resolution= 0.6,
    random_state=0,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    key_added='leiden_res_0_6'
)

In [ ]:
sc.pl.umap(adata, color=['leiden_res_0_6'], legend_loc = 'right margin', ncols = 2,  save = 'leiden_res_0_6.png'
          )

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_6", 
                        method="wilcoxon", layer = 'data')
sc.pl.rank_genes_groups_heatmap(adata, groupby="leiden_res_0_6", n_genes = 20, show_gene_labels=True, cmap="RdYlBu", dendrogram = True,
                               vmin = -2, vmax = 2, vcenter = 0, swap_axes = True, save = 'leiden_res_0_6_top20.png')

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_8", 
                        method="wilcoxon", layer = 'data')
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/home/yanjun2/scanpy/4/integrated/leiden_0_6.csv', index=False)

In [ ]:
sc.pl.rank_genes_groups_heatmap(adata, groupby="leiden_res_0_6", n_genes = 50, show_gene_labels=True, cmap="RdYlBu", dendrogram = True,
                               vmin = -2, vmax = 2, vcenter = 0, swap_axes = True, save = 'leleiden_res_0_6_top50.png')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=20, groupby="leiden_res_0_6", vmin = -1, vmax = 1, vcenter = 0,
                                  cmap = 'RdYlBu', swap_axes = True, save = 'leiden_res_0_6_top20.png')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=50, groupby="leiden_res_0_6", vmin = -1, vmax = 1, vcenter = 0,
                                  cmap = 'RdYlBu', swap_axes = True, save = 'leiden_res_0_6_top50.png')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=20, groupby="leiden_res_0_6", vmin = -1, vmax = 1, vcenter = 0,
                                  cmap = 'RdYlBu', swap_axes = True, save = 'leiden_res_0_6_top20.png')

In [ ]:
sc.pl.dotplot(adata, 'LYVE1', groupby = 'leiden_res_0_6', save = '0_6_LYVE1.png')

In [ ]:
# Save result
adata.write('/home/yanjun2/scanpy/4/integrated/integrated.h5ad')

In [ ]:
!pwd

In [ ]:
adata = sc.read('/Users/yanjun/Desktop/scanpy/7_integrated/integrated.h5ad')

In [ ]:
sc.pl.umap(adata, color=['leiden_res_0_8'], legend_loc = 'right margin')

In [ ]:
adata.obs["leiden_res_0_8"] = adata.obs["leiden_res_0_8"].map(
    {
        "0": "Mac0",
        "1": "Mac1",
        "2": "Mac2",
        "3": "Mac3",
        "4": "Mac4",
        "5": "Mac5",
        "6": "Mac5",
        "7": "Mac7"
    }
)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_8", method="wilcoxon", layer = 'data')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/Users/yanjun/Desktop/scanpy/7_integrated/DEGs/56_combined.csv', index=False)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_8", groups = ["Mac5", "Mac6"],
                        method="wilcoxon", layer = 'data')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/Users/yanjun/Desktop/scanpy/7_integrated/DEGs/5_vs_6.csv', index=False)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_8", groups = ["Mac5", "Mac0", "Mac1", "Mac2", "Mac3", "Mac4", "Mac7"],
                        method="wilcoxon", layer = 'data')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/Users/yanjun/Desktop/scanpy/7_integrated/DEGs/5_vs_l1neg.csv', index=False)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0_8", groups = ["Mac6", "Mac0", "Mac1", "Mac2", "Mac3", "Mac4", "Mac7"],
                        method="wilcoxon", layer = 'data')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv('/Users/yanjun/Desktop/scanpy/7_integrated/DEGs/6_vs_l1neg.csv', index=False)

In [ ]:
mdix = adata[adata.obs['condition'] == 'MDIX'].copy()

In [ ]:
mdi = adata[adata.obs['condition'] == 'MDI'].copy()

In [ ]:
obs_df = mdix.obs[['leiden_res_0_8', 'timepoint']]
frequency_df = obs_df.groupby(['timepoint', 'leiden_res_0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100

fig = plt.figure(figsize=(10, 6))
percentage_df.plot(kind='bar', stacked=True, ax=fig.gca())
plt.title("Percentage")
plt.xlabel("timepoint")
plt.ylabel("Percentage (%)")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

#fig.savefig("/home/yanjun2/scanpy/4/integrated/res06_percentage_plot.png")
plt.show()

In [ ]:
obs_df = mdi.obs[['leiden_res_0_8', 'timepoint']]
frequency_df = obs_df.groupby(['timepoint', 'leiden_res_0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100

fig = plt.figure(figsize=(10, 6))
percentage_df.plot(kind='bar', stacked=True, ax=fig.gca())
plt.title("Percentage")
plt.xlabel("timepoint")
plt.ylabel("Percentage (%)")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

#fig.savefig("/home/yanjun2/scanpy/4/integrated/res06_percentage_plot.png")
plt.show()

In [ ]:
obs_df = adata.obs[['leiden_res_0_8', 'condition']]
frequency_df = obs_df.groupby(['condition', 'leiden_res_0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100

fig = plt.figure(figsize=(10, 6))
percentage_df.plot(kind='bar', stacked=True, ax=fig.gca())
plt.title("Percentage")
plt.xlabel("Condition")
plt.ylabel("Percentage (%)")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

#fig.savefig("/home/yanjun2/scanpy/4/integrated/res06_percentage_plot.png")
plt.show()

In [ ]:
obs_df = adata.obs[['leiden_res_0_8', 'timepoint']]
frequency_df = obs_df.groupby(['timepoint', 'leiden_res_0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100

fig = plt.figure(figsize=(10, 6))
percentage_df.plot(kind='bar', stacked=True, ax=fig.gca())
plt.title("Percentage")
plt.xlabel("timepoint")
plt.ylabel("Percentage (%)")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

#fig.savefig("/home/yanjun2/scanpy/4/integrated/res06_percentage_plot.png")
plt.show()

In [ ]:
obs_df = adata.obs[['leiden_res_0_8', 'condition']]
frequency_df = obs_df.groupby(['condition', 'leiden_res_0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100

fig = plt.figure(figsize=(10, 6))
percentage_df.plot(kind='bar', stacked=True, ax=fig.gca())
plt.title("Percentage")
plt.xlabel("Condition")
plt.ylabel("Percentage (%)")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

fig.savefig("/home/yanjun2/scanpy/4/integrated/res08_percentage_plot.png")
plt.show()

In [ ]:
cell = ['CTGCCATAGTATGAAC-1_3']
umap = pd.DataFrame(adata.obsm['X_umap'][:,0:2], index=adata.obs_names, columns=['x', 'y'])

plt.scatter(umap["x"], umap["y"], s=5, color="lightgrey")
plt.scatter(umap.loc[cell, "x"], umap.loc[cell, "y"], s=30)

In [ ]:
# PAGA and diffusion maps
import scanpy.external as sce
def run_trajectory(data, clusterby, startcell):
    print('Running PAGA')
    sc.tl.paga(data, groups = clusterby)
    
    print('Calculating Palantir Diffusion maps')
    # using adaptive anisotropic adjacency matrix, instead of PCA projections (default) to compute diffusion components
    sce.tl.palantir(data, n_components = 25, knn = 50, impute_data = False, use_adjacency_matrix = True)
    
    print('Run Palantir')
    start = startcell
    pr_res = sce.tl.palantir_results(data, knn = 50, early_cell = start)
    # add pseudotime and differentiation potential information to anndata object
    data.obs['palantir_pseudotime'] = pr_res.pseudotime
    data.obs['palantir_entropy'] = pr_res.entropy
    
    run_trajectory.pr_res = pr_res.branch_probs

In [ ]:
run_trajectory(adata, clusterby = 'leiden_res_0_8', startcell = cell)

In [ ]:
sc.pl.embedding(adata, basis='X_palantir_diff_comp', color='condition', 
                    title='Palantir Diffusion Components', size=20)

In [ ]:
sc.pl.embedding(adata, basis='X_palantir_diff_comp', color='leiden_res_0_8', 
                    title='Palantir Diffusion Components', size=20)

In [ ]:
sc.pl.embedding(adata, basis='X_palantir_diff_comp', color='LYVE1', 
                    title='Palantir Diffusion Components', size=20)

In [ ]:
sc.pl.umap(adata, color='palantir_pseudotime', color_map='viridis', show=True)

In [ ]:
sc.tl.diffmap(adata)

In [ ]:
scv.pl.diffmap(adata, color = 'leiden_res_0_8', legend_loc = 'right margin')

In [ ]:
scv.pl.diffmap(adata, color = 'palantir_pseudotime', color_map='viridis')

In [ ]:
scv.pl.diffmap(adata, color = ['condition', 'LYVE1', 'leiden_res_0_8'], legend_loc = 'right margin')

In [ ]:
adata.write('/Users/yanjun/Desktop/scanpy/7_integrated/inte_palantir.h5ad')

In [ ]:
adata = sc.read('/Users/yanjun/Desktop/scanpy/7_integrated/inte_palantir.h5ad')

In [ ]:
!pwd

In [ ]:
from scipy.stats import pearsonr

bulk_data = pd.read_csv('/Users/yanjun/Desktop/scanpy/decon/mdix.csv', index_col=0)

common_genes = bulk_data.index.intersection(adata.var_names)
bulk_data = bulk_data.loc[common_genes]
adata = adata[:, common_genes]

pseudo_bulk_profiles = adata.to_df().groupby(adata.obs['leiden_res_0_8']).mean()

correlation_results = {}

for cluster, cluster_profile in pseudo_bulk_profiles.iterrows():
    correlation, _ = pearsonr(cluster_profile, bulk_data.mean(axis=1))  

correlation_df = pd.DataFrame.from_dict(correlation_results, orient='index', columns=['Correlation'])
correlation_df = correlation_df.sort_values(by='Correlation', ascending=False)
print(correlation_df)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.bar(correlation_df.index.astype(str), correlation_df['Correlation'], color='skyblue')
plt.xlabel('Cluster')
plt.ylabel('Correlation with Bulk RNA-seq')
plt.title('Correlation between Clusters and Bulk RNA-seq Data')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sc.pl.violin(adata, keys=["palantir_pseudotime"], groupby="leiden_res_0_8", rotation=90, legend = 'auto')

In [ ]:
sc.pl.umap(mdi, color='palantir_pseudotime', color_map='viridis', show=True)

In [ ]:
sc.pl.umap(mdix, color='palantir_pseudotime', color_map='viridis', show=True)

In [ ]:
sc.pl.umap(adata, color = 'palantir_pseudotime')

In [ ]:
adata = sc.read('/home/yanjun2/scanpy/7/inte_palantir.h5ad')

In [ ]:
ldata1 = scv.read('/home/yc/Desktop/loom/R1_D1MDI_D1A.loom', cache=True)
ldata2 = scv.read('/home/yc/Desktop/loom/R1_D2MDI_D2A.loom', cache=True)
ldata3 = scv.read('/home/yc/Desktop/loom/R1_D3MDI_D3A.loom', cache=True)
ldata4 = scv.read('/home/yc/Desktop/loom/R1_D4MDI_D4A.loom', cache=True)
ldata5 = scv.read('/home/yc/Desktop/loom/R1_D5MDI_D5A.loom', cache=True)
#ldata6 = scv.read('/home/yc/Desktop/loom/R1_DMo_D0M.loom', cache=True)

In [ ]:
def clean_cell_ids(ldata):
    ldata.obs.index = ldata.obs.index.str.replace(r'^.*:', '', regex=True)
    ldata.obs.index = ldata.obs.index.str.replace(r'x$', '', regex=True)
    return ldata
ldata1 = clean_cell_ids(ldata1)
ldata2 = clean_cell_ids(ldata2)
ldata3 = clean_cell_ids(ldata3)
ldata4 = clean_cell_ids(ldata4)
ldata5 = clean_cell_ids(ldata5)

In [ ]:
ldata1.var_names_make_unique()
ldata2.var_names_make_unique()
ldata3.var_names_make_unique()
ldata4.var_names_make_unique()
ldata5.var_names_make_unique()
#ldata6.var_names_make_unique()

In [ ]:
ldata = ldata1.concatenate([ldata2, ldata3, ldata4, ldata5
                            ])

In [ ]:
adata.obs_names = adata.obs_names.str.split('-').str[0]

In [ ]:
adata = scv.utils.merge(adata, ldata)

In [ ]:
adata.obs['hash.ID'].unique().tolist()

In [ ]:
sc.tl.paga(adata, 'leiden_res_1')
scv.pl.paga(adata, basis='umap', threshold=0.4, max_edge_width=0.3, alpha=.4, title='',  
            figsize=(4.7,5), legend_loc='right margin')

In [ ]:
adata.X = adata.layers['spliced']
adata.X = adata.layers['unspliced']

In [ ]:
scv.pp.filter_and_normalize(adata)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=0)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
##scv.tl.recover_dynamics(adata, n_jobs=8)

In [ ]:
scv.tl.velocity(adata, mode = 'stochastic')

In [ ]:
scv.tl.velocity_graph(adata)

In [ ]:
scv.pl.velocity_embedding(adata, basis='umap',color = ['leiden_res_0_8'])

In [ ]:
sc.pl.violin(adata, keys=["palantir_pseudotime"], groupby="leiden_res_0_8", rotation=90, legend = 'auto', save = 'leiden08_palantirpt.png')

In [ ]:
adata.obsm['exp_diffmap'] = adata.obsm['X_diffmap']

In [ ]:
sc.tl.diffmap(adata, n_comps = 30)
scv.pl.diffmap(adata, color='leiden_res_0_8', legend_loc='right margin', figsize=(10,10), 
               size=20, alpha=0.8, title= 'Diffusion Map'#, save = 'MDIXwo_diffmap_ct.pdf'
              )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='exp_diffmap', color = 'leiden_res_0_8'#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'leiden_res_0_8'#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap', color = 'leiden_res_0_8'#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap', color = 'leiden_res_0_8', legend_loc='right margin',
                                 size = 100#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', cmap='gnuplot')

In [ ]:
scv.pl.scatter(adata, color='palantir_pseudotime', cmap='gnuplot')

In [ ]:
sc.pl.scatter(adata, x = 'palantir_pseudotime', y = 'velocity_pseudotime')

In [ ]:
scv.pl.velocity(adata, ['DPYD', 'ELMO1', 'MALAT1', 'NEAT1', 'AP2A2', 'F13A1', 'MYO5A', 'HDAC', 'SLC9A9','MS4A4E',
                        'GRN1', 'STAB1', 'PSAP', 'APOE', 'APOC1', 'ACE', 'LYZ'
], ncols=2, basis = 'diffmap')

In [ ]:
##cellrank

In [ ]:
adata.X = adata.layers['spliced']
adata.X = adata.layers['unspliced']

In [ ]:
scv.pp.filter_and_normalize(adata)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=0)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
scv.tl.recover_dynamics(adata, n_jobs=8)

In [ ]:
scv.tl.velocity(adata, mode = 'stochastic')

In [ ]:
from cellrank.kernels import VelocityKernel
from cellrank.kernels import ConnectivityKernel

In [ ]:
vk = cr.kernels.VelocityKernel(adata)

In [ ]:
vk.compute_transition_matrix()

In [ ]:
vk.plot_projection()

In [ ]:
vk.plot_projection()

In [ ]:
ck = cr.kernels.ConnectivityKernel(adata)
ck.compute_transition_matrix()

combined_kernel = 0.8 * vk + 0.2 * ck

In [ ]:
combined_kernel.plot_random_walks(start_ixs={"leiden_res_0_8": "Mac7"}, max_iter=200, seed=0)

In [ ]:
ck.plot_projection(basis = 'diffmap', stream = True)

In [ ]:
combined_kernel.plot_projection()

In [ ]:
from cellrank.estimators import GPCCA
g = GPCCA(vk)
print(g)

In [ ]:
g.fit(n_states=10, cluster_key="leiden_res_0_8")
g.plot_macrostates(which="all")

In [ ]:
g.predict_terminal_states(method="top_n", n_states=6)
g.plot_macrostates(which="terminal")

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(legend_loc="right")

In [ ]:
cr.pl.circular_projection(adata, keys="leiden_res_0_8", legend_loc="right")

In [ ]:
mono_drivers = g.compute_lineage_drivers(lineages="Mono_1_1")
mono_drivers.head(10)

In [ ]:
model = cr.models.GAMR(adata)

In [ ]:
cr.pl.gene_trends(
    adata,
    model=model,
    data_key="data",
    genes=["C1QA", "C1QC", "LYVE1", "F13A1"],
    same_plot=True,
    ncols=2,
    time_key="palantir_pseudotime",
    hide_cells=True,
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    data_key="data",
    genes=["C1QA", "C1QC", "LYVE1", "F13A1"],
    lineages=["Mono_1_1", "Mega", "CLP"],
    time_key="palantir_pseudotime",
    cbar=False,
    show_all_genes=True,
)

In [ ]:
#### cytotrace ###

In [ ]:
adata.layers["spliced"] = adata.X
adata.layers["unspliced"] = adata.X
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

In [ ]:
from cellrank.kernels import CytoTRACEKernel

ctk = CytoTRACEKernel(adata).compute_cytotrace()

In [ ]:
scv.pl.scatter(
    adata,
    color=["ct_pseudotime", "velocity_pseudotime", "palantir_pseudotime"],
    basis="diffmap",
    color_map="gnuplot2",
)

In [ ]:
sc.pl.umap(adata, color = 'ct_pseudotime')

In [ ]:
sc.pl.violin(adata, keys=["ct_pseudotime"], groupby="leiden_res_0_8", rotation=90, legend = 'auto')